## **Physics-Informed Neural Networks for Spherical Brane-Skyrmions**

**Author:** Sergio Sánchez Rentero  
**Contact:** sergis15@ucm.es  
**Links:** [GitHub Profile](https://github.com/sergiosr150703-lang)  

**Project Context:** This code is part of a Master's Thesis in Theoretical Physics. The project investigates the canonical quantization of topological solitons in brane-world scenarios. Specifically, this notebook implements Physics-Informed Neural Networks (PINNs) using PyTorch to optimize the energy functional, extract the scalar profiles of spherical Brane-Skyrmions, and perform a experimental fit of the free parameters.

**Last Updated:** June 2026

# <span style="color: #00d0ff;">Neural Network configuration</span>

In [2]:
## Libraries ##

import numpy as np # Numerical computing library

import torch # PyTorch library
import torch.nn as nn # Neural network module
import torch.optim as optim # Optimization algorithms

import matplotlib.pyplot as plt # Plotting library
from IPython import display # IPython display utilities (e.g., for animations)
import matplotlib.gridspec as gridspec # Grid specification for subplots

from matplotlib import animation # Animation support
from IPython.display import HTML # HTML display in IPython

import ipywidgets as widgets # Interactive widgets for Jupyter notebooks

In [3]:
# Check for MPS (Metal Performance Shaders) availability on macOS
if not torch.backends.mps.is_available():
    if not torch.backends.mps.is_built():
        print("MPS not available because the current PyTorch install was not "
              "built with MPS enabled.")
    else:
        print("MPS not available because the current MacOS version is not 12.3+ "
              "and/or you do not have an MPS-enabled device on this machine.")

else:
    mps_device = torch.device("mps")

MPS not available because the current PyTorch install was not built with MPS enabled.


In [4]:
## Auxiliary functions for computing derivatives and generating grids ##

def gradients(outputs, inputs, order = 1): # Function for computing derivatives of order 'order'
    if order == 1:
        return torch.autograd.grad(outputs, inputs, grad_outputs=torch.ones_like(outputs), create_graph=True)[0] # First derivative
    elif order > 1:
        return gradients(gradients(outputs, inputs, 1), inputs, order - 1) # Recursive call for higher-order derivatives
    else:
        return outputs

def generate_2Dgrid(range1, range2, Ns, requires_grad = True): # Function for generating a 2D grid of points for neural network evaluation
    grid1 = np.linspace(range1[0], range1[1], Ns[0], dtype = np.float32) # Generate uniformly spaced points in the first range
    grid2 = np.linspace(range2[0], range2[1], Ns[1], dtype = np.float32) # Generate uniformly spaced points in the second range
    x0, y0 = np.meshgrid(grid1, grid2) # Create a 2D grid from the generated points
    x = torch.tensor(x0.reshape(Ns[0]*Ns[1], 1), requires_grad = requires_grad) # Convert the grid to PyTorch tensors
    y = torch.tensor(y0.reshape(Ns[0]*Ns[1], 1), requires_grad = requires_grad) # Convert the grid to PyTorch tensors
    return x, y

def plot_form(x, y, z): # Function for plotting the results in 2D of a neural network
    Nx  = x.unique().shape[0] # NNumber of unique points in the x direction
    Ny = y.unique().shape[0] # NNumber of unique points in the y direction
    return map(lambda t: t.reshape(Ny, Nx).cpu().detach().numpy(), (x, y, z)) # Reshape and conversion to numpy

In [5]:
## Configuration of PyTorch to use all available CPU threads for computations ##
import os # Library for interacting with the operating system
torch.set_num_threads(os.cpu_count()) # Configure PyTorch to use all available CPU cores

print("Using", torch.get_num_threads(), "threads on CPU") # Print the number of threads that PyTorch is using on the CPU

Using 12 threads on CPU


## Modules and functionals
Configuration of the modules and functionals for both Atiyah-Manton ansatz and the general case

In [ ]:
torch.set_default_dtype(torch.float64) # Double precision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Check if CUDA (GPU) is available and set the device accordingly
print(f"Usando dispositivo: {device}")

# Model for minimizing one parameter (L) for the Atiyah-Manton ansatz
class AtiyahMantonSkyrmion(nn.Module):
    def __init__(self, initial_L=0.5):
        super().__init__()
        self.log_L = nn.Parameter(torch.tensor(np.log(initial_L), dtype=torch.float64))
        # log and exp ensure L > 0

    def forward(self, r):
        L = torch.exp(self.log_L)
        f = torch.pi * (1 - r / torch.sqrt(r**2 + L**2)) # Atiyah-Manton ansatz
        
        return f, L

# Functional of energy to minimize
def energy_functional(model, physics, r, theta, r_1d, theta_1d, lam, J):
    f, L = model(r)
    total_energy, radius, H0, H2, H4 = physics(f, r, theta, r_1d, theta_1d, lam, J)
    return total_energy, radius, L, H0, H2, H4

# Model for the general Neural Network
class ODE(nn.Module):
    def __init__(self):
        super(ODE, self).__init__()

        self.net = nn.Sequential( #Set of layers of the network, and transformations on each layer
            nn.Linear(1, 40), # 40 learnable parameters
            nn.GELU(), #nonlinear transformation, no learnable parameters
            nn.Linear(40, 40), #40x40 learnable parameters
            nn.GELU(),
            nn.Linear(40, 1), # Another 40
        )

        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean = 0, std = 0.1) #initialization of learnable parameters (weights and biases)
                nn.init.constant_(m.bias, val = 0.0)
                nn.init.constant_(self.net[4].weight, val=0.0)
                nn.init.constant_(self.net[4].bias, val=0.0)

    # Output of the network
    def forward(self, r, optimal_L, max_r):
        Ansatz = torch.pi * (1 - r / torch.sqrt(r**2 + optimal_L**2)) # Atiyah-Manton ansatz
        epsilon = 1e-5  # Small parameter
        V = (optimal_L/(optimal_L+r)) * (torch.exp(-epsilon/r**2) * torch.exp(-epsilon/((max_r-epsilon)-r)**2)) * (r>0) * (r<max_r) # Variation ansatz

        #V = ((max_r - r)/max_r) * (2*optimal_L*r/(r**2 + optimal_L**2))**2 # Variation ansatz alternative
        #V = ((max_r - r)/max_r) * (r/optimal_L) * torch.exp(1-r/optimal_L) # Variation ansatz alternative 2

        correction = self.net(r) # Correction given by the neural network
        F = Ansatz + V * correction # Dynamic ansatz: F(r) = Ansatz(r) + V(r) * NN(r)
        return F

# Loss function based on the energy functional, which is what we want to minimize
class ODELoss(nn.Module):
    def __init__(self, ode):
        super(ODELoss, self).__init__()
        self.ode = ode 

    def forward(self, physics, r, theta, r_1d, theta_1d, lam, J, optimal_L, max_r):
        f  = self.ode(r, optimal_L, max_r) # The profile is given by the NN
        total_energy, radius, H0, H2, H4 = physics(f, r, theta, r_1d, theta_1d, lam, J) # The energy is computed by the physics module
        return total_energy, radius, H0, H2, H4

Usando dispositivo: cpu


# <span style="color: #00d0ff;">Physical configurations</span>

For all configurations: The boundary conditions are $F(0)=\pi$ and $F(\infty)=0$. The expressions hold for dimension-less quantities. We will want to compute the normalized energy: $M/2\pi^2$.

The isoscalar root mean square radius is:
$$\langle r^2\rangle=-\frac{2}{\pi}\int dr \; r^2 F'(r)\sin^2 F(r).$$

The Atiyah-Manton ansatz for spherical Skyrmions is:
$$F(r)=\pi\left(1-\frac{1}{\sqrt{1+(L/r)^2}}\right).$$

 ## Static configuration with curvature

The non-normalized energy functional for the static configuration of the spherical Brane-Skyrmion is:
$$M[F]=4\pi\int dr \Big[(1-\lambda R)\frac{\sqrt{g}}{\sin\theta} - r^2\Big],$$
where:
$$\frac{\sqrt{g}}{\sin\theta}=\sqrt{1+F'^2(r)}\rho^2,\quad \rho^2(r)=r^2+\sin^2 F(r),$$
$$R=-\frac{2}{\rho^2}\Big(1-\frac{1}{A}\Big)-\frac{2 A'}{A^2\rho\rho'},\quad A(r)=\frac{1+F'^2(r)}{\rho'^2}.$$


In [21]:
def physics(f, r, theta, r_1d, theta_1d, lam, J):

    df = gradients(f, r, 1) # F'
    rho = torch.sqrt(r**2 + torch.sin(f)**2) # rho
    metric = (rho**2)*torch.sqrt(1+df**2) # sqrt{g}/sin{theta}
    drho = gradients(rho, r, 1) # rho'
    A = (1+df**2)/(drho**2) # A
    dA = gradients(A, r, 1) # A'
    R = - (2/rho**2)*(1-1/A) - (2*dA/((A**2)*rho*drho)) # Scalar curvature

    # Total energy normalized by 2*pi^2
    H0 = 4*torch.pi*torch.trapz(((1 - lam * R) * metric - r**2).squeeze(), r_1d.squeeze())/(2*torch.pi**2)
    H2 = torch.tensor(0.0, dtype=torch.float64, device=r.device)
    H4 = torch.tensor(0.0, dtype=torch.float64, device=r.device)
    radius = torch.sqrt(-(2/torch.pi)*torch.trapz((df*(r*torch.sin(f))**2).squeeze(), r_1d.squeeze()))
    total_energy = H0
    return total_energy, radius, H0, H2, H4

## Dynamic configuration without curvature

Up to order $J^4$, the Hamiltonian as a functional of $F(r)$ is:
$$H[F;J]=\alpha+\frac{1}{4\beta}J^2-\frac{\gamma}{4\beta^4}J^4+\mathcal{O}(J^6),$$
where:
$$\alpha=4\pi\int dr \left[\frac{\sqrt{g}}{\sin\theta}-r^2\right],\quad \beta=\frac{4\pi}{3}\int dr \frac{\sqrt{g}}{\sin\theta}C(r),\quad \gamma=\frac{4\pi}{15}\int dr \frac{\sqrt{g}}{\sin\theta}C^2(r),$$
and where:
$$\frac{\sqrt{g}}{\sin\theta}=\sqrt{1+F'^2(r)}[r^2+\sin^2 F(r)],\quad C(r)=\frac{r^2\sin^2 F(r)}{r^2+\sin^2 F(r)}.$$

In [ ]:
def physics(f, r, theta, r_1d, theta_1d, lam, J):
    
    df = gradients(f, r, 1) # F'
    rho = torch.sqrt(r**2 + torch.sin(f)**2) # rho
    metric = (rho**2)*torch.sqrt(1+df**2) # sqrt{g}/sin{theta}
    C = (r*torch.sin(f))**2/(r**2+torch.sin(f)**2) # C

    # Integrals for the coefficients
    alpha = 4*torch.pi*torch.trapz((metric - r**2).squeeze(), r_1d.squeeze())
    beta = (4*torch.pi/3)*(torch.trapz((metric*C).squeeze(), r_1d.squeeze()))
    gamma = (4*torch.pi/15)*(torch.trapz((metric*C**2).squeeze(), r_1d.squeeze()))

    # Total energy normalized by 2*pi^2
    H0 = alpha/(2*torch.pi**2)
    H2 = (1/(4*beta))/(2*torch.pi**2)
    H4 = (-gamma/(16*beta**4))/(2*torch.pi**2)
    radius = torch.sqrt(-(2/torch.pi)*torch.trapz((df*(r*torch.sin(f))**2).squeeze(), r_1d.squeeze()))
    total_energy = H0+H2*J**2+H4*J**4
    return total_energy, radius, H0, H2, H4

## Dynamic configuration with curvature

Up to order $J^4$ and $\omega^4$, the Hamiltonian and the scalar curvature are:
$$H[F;J]=\alpha+\frac{1}{4\beta}J^2-\frac{\gamma}{4\beta^4}J^4+\mathcal{O}(J^6),\quad R=R_0+R_2\omega^2+R_4\omega^4+\mathcal{O}(\omega^6),$$
where:
$$\alpha=4\pi\int dr \left[(1-\lambda R_0)\frac{\sqrt{g_0}}{\sin\theta}-r^2\right],\quad \beta=2\pi\int d\theta\int dr \left[\frac{1}{2}(1-\lambda R_0)C\sin^2\theta+\lambda R_2\right]\sqrt{g_0},$$
$$\gamma=2\pi\int d\theta\int dr \left[\frac{1}{8}(1-\lambda R_0)C^2\sin^4\theta-\frac{1}{2}\lambda R_2 C\sin^2\theta+\lambda R_4\right]\sqrt{g_0},$$
$$\sqrt{g_0}=\sqrt{1+F'^2(r)}[r^2+\sin^2 F(r)]\sin\theta,\quad C(r)=\frac{r^2\sin^2 F(r)}{r^2+\sin^2 F(r)}.$$

In [7]:
def physics(f, r, theta, r_1d, theta_1d, lam, J):
    
    df = gradients(f, r, 1) # F'
    rho = torch.sqrt(r**2 + torch.sin(f)**2) # rho
    metric = (rho**2)*torch.sqrt(1+df**2)*torch.sin(theta) # sqrt{g}
    drho = gradients(rho, r, 1) # rho'
    ddrho = gradients(drho, r, 1) # rho''
    B = (1+df**2) # B
    dB = gradients(B, r, 1) # B'
    Lambda = torch.sin(f)**2 # Lambda
    dLambda = gradients(Lambda, r, 1) # Lambda'
    ddLambda = gradients(dLambda, r, 1) # Lambda''
    C = (r*torch.sin(f))**2/(r**2+torch.sin(f)**2) # C

    # Curvature coefficients
    R0 = -(2*(B**2+rho*dB*drho-B*(drho**2+2*rho*ddrho)))/((B**2)*(rho**2))

    R2 = (-2*(B**2)*(1+3*torch.cos(2*theta))*Lambda*(r**2)
        +rho*dB*(-2*Lambda*rho*dLambda+(rho**3)*dLambda+2*(Lambda**2)*drho)*(torch.sin(theta)**2)
        -B*rho*(torch.sin(theta)**2)*(4*(rho**2)*dLambda*drho+2*(rho**3)*ddLambda-rho*(3*(dLambda**2)
        +4*Lambda*ddLambda)+4*Lambda*(dLambda*drho+Lambda*ddrho)))/(2*(B**2)*(rho**4))
    
    R4 = (1/(2*(B**2)*(rho**6)))*(-4*(B**2)*(1+2*torch.cos(2*theta))*(torch.sin(theta)**2)*(Lambda**2)*(r**4)
                                +(torch.sin(theta)**4)*Lambda*rho*(r**2)*dB*(-2*Lambda*rho*dLambda+(rho**3)*dLambda+2*(Lambda**2)*drho)
                                +B*(torch.sin(theta)**4)*(-(rho**6)*(dLambda**2)+Lambda*(rho**4)*(7*(dLambda**2)-4*rho*dLambda*drho-2*(rho**2)*ddLambda)
                                                          +(Lambda**2)*(rho**2)*(-7*(dLambda**2)-4*rho*dLambda*drho+6*(rho**2)*ddLambda)
                                                          -4*(Lambda**4)*(drho**2-rho*ddrho)-4*(Lambda**3)*rho*(-3*dLambda*drho+rho*(ddLambda+rho*ddrho))))
    
    # Integrals in theta and r for the coefficients
    alpha_theta = torch.trapz(((1 - lam*R0)*metric - torch.sin(theta)*r**2), theta_1d, dim=1)
    alpha = 2*torch.pi*torch.trapz(alpha_theta.squeeze(), r_1d.squeeze())
    beta_theta = torch.trapz((C*(1-lam*R0)*(torch.sin(theta)**2)/2 + lam*R2)*(metric), theta_1d, dim=1)
    beta = 2*torch.pi*(torch.trapz(beta_theta.squeeze(), r_1d.squeeze()))
    gamma_theta = torch.trapz(((C**2)*(1-lam*R0)*(torch.sin(theta)**4)/8 - (lam*R2*C*(torch.sin(theta)**2))/2 + lam*R4)*(metric), theta_1d, dim=1)
    gamma = 2*torch.pi*(torch.trapz(gamma_theta.squeeze(), r_1d.squeeze()))

    # Total energy normalized by 2*pi^2
    H0 = alpha/(2*torch.pi**2)
    H2 = (1/(4*beta))/(2*torch.pi**2)
    H4 = (-gamma/(16*beta**4))/(2*torch.pi**2)
    radius = torch.sqrt(-(2/torch.pi)*torch.trapz((df*(r*torch.sin(f))**2).squeeze(), r_1d.squeeze()))
    total_energy = H0+H2*J**2+H4*J**4
    return total_energy, radius, H0, H2, H4

# <span style="color: #00d0ff;">Optimization loops</span>

## Atiyah-Manton optimization
This cell has to be run before the general case, since the later will make use of the optimal $L$ found here.

In [28]:
# Numerical parameters for both cells
lam = 1 # Lambda for the curvature term
J = 0 # Angular momentum
max_r = 25 # Higher integration limit for r
min_r = 1e-10 # Lower integration limit
box = 10 # Visualization size
Nr = 2000 # NNumber of points in r
max_theta = np.pi # Higher integration limit for theta
min_theta = 1e-5 # Lower integration limit for theta
Ntheta = 100 # Number of points in theta

# Grid of points for the optimization
r_numpy = np.geomspace(min_r, max_r, Nr) # Grid of points in r (logarithmic spacing)
r_grid_1d = torch.tensor(r_numpy, dtype=torch.float64, device=device)
r_grid = r_grid_1d.reshape(-1, 1)
r_grid.requires_grad = True

theta_numpy = np.linspace(min_theta, max_theta, Ntheta) # Grid of points in theta (linear spacing)
theta_grid_1d = torch.tensor(theta_numpy, dtype=torch.float64, device=device)
theta_grid = theta_grid_1d.reshape(1, -1)

# Initialize the model, optimizer, and lists to store the history of energy and parameters
model = AtiyahMantonSkyrmion(initial_L=0.1).to(device) 
optimizer = optim.Adam(model.parameters(), lr=0.1) 

energy_nucleon_history = []
r_nucleon_history = []
L_nucleon_history = []
H0_history = []
H2_history = []
H4_history = []

# Two separate figures for the loss and the solution
fig1, ax1 = plt.subplots(figsize=(8, 5))
fig2, ax2 = plt.subplots(figsize=(8, 5))

plt.close(fig1)
plt.close(fig2)

font = {'size': 14} 
plt.rc('font', **font)

# Widgest for displaying the two figures separately
out1 = widgets.Output()
out2 = widgets.Output()
graphics_panel = widgets.HBox([out1, out2]) # For displaying the two figures side by side

display.display(graphics_panel)

# Optimization loop for the nucleon
print(f"\n--- OPTIMIZATION ---")
for epoch in range(501):
    try:
        optimizer.zero_grad() # Resets the accumulated gradients

        # Compute the energy and other functionals
        current_E, current_r, current_L, current_H0, current_H2, current_H4 = energy_functional(model, physics, r_grid, theta_grid, r_grid_1d, theta_grid_1d, lam, J)
    
        current_E.backward()
        optimizer.step()
    
        energy_nucleon_history.append(current_E.item())
        r_nucleon_history.append(current_r.item())
        L_nucleon_history.append(current_L.item())
        H0_history.append(current_H0.item())
        H2_history.append(current_H2.item())
        H4_history.append(current_H4.item())

        if epoch % 100 == 0: # Print the current energy and parameters every 100 epochs
            print(f"Epoch {epoch}: E = {current_E.item():.5f} | r = {current_r.item():.5f} | L = {current_L.item():.5f}")

            # Loss plot
            ax1.cla()
            ax1.set_xlabel('Epoch')
            ax1.set_ylabel('Parameter L (loss function)')
            ax1.set_yscale('log')
            ax1.plot(L_nucleon_history)
            #ax1.set_title('Training convergence of L')
            ax1.grid(True, which="both", ls="--")

            # Soliton profile plot
            ax2.cla()
            with torch.no_grad():
                f_final, _ = model(r_grid)
            ax2.set_xlabel('r')
            ax2.set_ylabel('F(r)')
            ax2.plot(r_grid.cpu().detach().numpy(), f_final.cpu().detach().numpy(),label=rf'$\lambda = {lam}$, $J = {J}$')
            ax2.set_xlim(0, box)
            #ax2.set_title('Soliton profile')
            ax2.grid(True, which="both", ls="--")
            ax2.legend()

            # Display the two figures in their respective output widgets
            with out1:
                display.clear_output(wait=True)
                display.display(fig1)
                
            with out2:
                display.clear_output(wait=True)
                display.display(fig2)
                
    except KeyboardInterrupt:
        break

# Results
optimal_L = L_nucleon_history[-1]
print(f"\n--- RESULTS ---")
print(f"L = {optimal_L:.5f}")
print(f"E = {energy_nucleon_history[-1]:.5f}")
print(f"r = {r_nucleon_history[-1]:.5f}")
print(f"H0 = {H0_history[-1]:.5f}")
print(f"H2 = {H2_history[-1]:.3e}")
print(f"H4 = {H4_history[-1]:.3e}")


--- OPTIMIZATION ---
Epoch 0: E = 5.54391 | r = 0.07465 | L = 0.10000


Epoch 100: E = 2.99114 | r = 0.93431 | L = 1.25168


Epoch 200: E = 2.99102 | r = 0.92531 | L = 1.23962


Epoch 300: E = 2.99102 | r = 0.92536 | L = 1.23969


Epoch 400: E = 2.99102 | r = 0.92536 | L = 1.23969


Epoch 500: E = 2.99102 | r = 0.92536 | L = 1.23969



--- RESULTS ---
L = 1.23969
E = 2.99102
r = 0.92536
H0 = 2.99102
H2 = 0.000e+00
H4 = 0.000e+00


## General optimization
This cell requires the optimal $L$ of the Atiyah-Manton optimization to initialize the neural network.

In [ ]:
ode = ODE().to(device) # Moves the neural network to the configured device (GPU or CPU)
odeloss = ODELoss(ode) # Loss function based on the energy functional, which is what we want to minimize

# Lists to store the history of energy and parameters during training
energy_nucleon_history = [] 
r_nucleon_history = []
H0_history = []
H2_history = []
H4_history = []

optimizer = optim.Adam(ode.parameters(), lr=8e-3) # Adam optimizer for training the neural network parameters
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=(0.01)**(1/2000)) # Scheduler for the learning rate

# Two separate figures for the loss and the solution
fig1, ax1 = plt.subplots(figsize=(8, 5))
fig2, ax2 = plt.subplots(figsize=(8, 5))

plt.close(fig1)
plt.close(fig2)

font = {'size': 14}
plt.rc('font', **font)

# Widgest for displaying the two figures separately
out1 = widgets.Output()
out2 = widgets.Output()
graphics_panel = widgets.HBox([out1, out2]) # For displaying the two figures side by side

display.display(graphics_panel)

# We import data from Mathematica for comparison with the shooting method, if available
# Make sure the CSV file is in the same folder as your .ipynb
import numpy as np

try:
    data_shoot = np.loadtxt("Profile_Shooting_Lambda_1.csv", delimiter=",")
    r_shoot = data_shoot[:, 0]
    F_shoot = data_shoot[:, 1]
    have_shooting = True
    print("Shooting data loaded.")
except FileNotFoundError:
    have_shooting = False
    print("Shooting data file not found. It will be omitted from the plot.")

# Optimization loop
print(f"\n--- OPTIMIZATION ---")
for epoch in range(2001):
    try:
        optimizer.zero_grad() # Resets the accumulated gradients

        # Compute the energy and other functionals using the ODELoss, which internally uses the ODE model
        current_E, current_r, current_H0, current_H2, current_H4 = odeloss(physics, r_grid, theta_grid, r_grid_1d, theta_grid_1d, lam, J, optimal_L, max_r)
        current_E.backward() # Computes the gradients of the loss with respect to the neural network parameters
        optimizer.step() # Updates the parameters of the neural network according to the calculated gradients

        scheduler.step() # Updates the learning rate according to the scheduler

        # Store the current energy and parameters in the history lists
        energy_nucleon_history.append(current_E.item())
        r_nucleon_history.append(current_r.item())
        H0_history.append(current_H0.item())
        H2_history.append(current_H2.item())
        H4_history.append(current_H4.item())

        if epoch % 100 == 0: # Print the current energy and parameters every 100 epochs
            
            print(f"Epoch {epoch}: E = {current_E.item():.5f} | r = {current_r.item():.5f} | H0 = {current_H0.item():.5f} | H2 = {current_H2.item():.3e} | H4 = {current_H4.item():.3e}")
            # Loss plot
            ax1.cla()
            ax1.set_xlabel('Epoch')
            ax1.set_ylabel('Energy (Loss function)')
            ax1.set_yscale('log')
            ax1.plot(energy_nucleon_history)
            ax1.grid(True, which="both", ls="--")

            # Soliton profile plot
            ax2.cla()
            with torch.no_grad():
                f_final, _ = model(r_grid)
                f_NN_final = ode(r_grid, optimal_L, max_r)
            ax2.set_xlabel('r')
            ax2.set_ylabel('F(r)')
            ax2.plot(r_grid.cpu().detach().numpy(), f_NN_final.cpu().detach().numpy(), label=rf'PINN ($\lambda = {lam}$, $J = {J}$)', linewidth=2.5)
            ax2.plot(r_grid.cpu().detach().numpy(), f_final.cpu().detach().numpy(), label='Atiyah-Manton ansatz', color='black', linestyle='--')
            # Uncomment the following lines if you have shooting data for comparison
            #if have_shooting:
            #    ax2.plot(r_shoot, F_shoot, label='Shooting Method', color='tab:red', linestyle='--', linewidth=2.5)
            ax2.set_xlim(0, box)
            ax2.grid(True, which="both", ls="--")
            ax2.legend()

            # Update the two figures in their respective output widgets
            with out1:
                display.clear_output(wait=True)
                display.display(fig1)
                
            with out2:
                display.clear_output(wait=True)
                display.display(fig2)

    except KeyboardInterrupt:
        break

# Data export to CSV for comparison with other software
r_export = r_grid.cpu().detach().numpy().squeeze()
F_export = f_NN_final.cpu().detach().numpy().squeeze()
data_export = np.column_stack((r_export, F_export))
doc_name = f"Profile_NN_lam_{lam}_J_{J}.csv"
np.savetxt(doc_name, data_export, delimiter=",", header="r,F", comments='')
print(f"\n Data exported to '{doc_name}'")

# Results
print(f"\n--- RESULTS ---")
print(f"E = {energy_nucleon_history[-1]:.5f}")
print(f"r = {r_nucleon_history[-1]:.5f}")
print(f"H0 = {H0_history[-1]:.5f}")
print(f"H2 = {H2_history[-1]:.3e}")
print(f"H4 = {H4_history[-1]:.3e}")

Datos de Mathematica cargados con éxito.

--- OPTIMIZATION ---
Epoch 0: E = 2.99102 | r = 0.92536 | H0 = 2.99102 | H2 = 0.000e+00 | H4 = 0.000e+00


Epoch 100: E = 2.95582 | r = 0.87553 | H0 = 2.95582 | H2 = 0.000e+00 | H4 = 0.000e+00


Epoch 200: E = 2.95524 | r = 0.88232 | H0 = 2.95524 | H2 = 0.000e+00 | H4 = 0.000e+00



 Data exported to 'Perfil_NN_lam_1_J_0.csv'

--- RESULTS ---
E = 2.95523
r = 0.88050
H0 = 2.95523
H2 = 0.000e+00
H4 = 0.000e+00


# <span style="color: #00d0ff;">Experimental fit</span>

Dimension-less and rescaled quantities are:
$$\lambda^*=\frac{\lambda}{f^2 R_B^2},\quad J^*=\frac{J}{f^4 R_B^3},\quad E^*=\frac{E}{f^4 R_B^3},\quad r^*=\frac{\sqrt{\langle r^2\rangle}}{R_B},$$
where $[f]=\text{MeV}$ and $[R_B]=\text{MeV}^{-1}=\text{fm}$.

The experimental ratio to fit (for a given value of $\lambda^*$) is:
$$\frac{M_N r_N}{J_N}=\frac{E^*(J_N^*,\lambda^*)r^*(J_N^*,\lambda^*)}{J_N^*},$$
where $J_N=\sqrt{j_N(j_N+1)}=\sqrt{3/4}$, with $j_N=1/2$, $M_N\approx 939\text{MeV}$ and $r_N\approx 0.72\text{fm}$.

The rest of the free parameters are obtained via algebraic substitution:
$$R_B=\frac{r_N}{r_N^*},\quad f=\frac{1}{R_B}\left(\frac{J_N}{J_N^*}\right)^{1/4},\quad \lambda=f^2 R_B^2\lambda^*.$$

In [ ]:
# Parameters
max_r = 20
min_r = 1e-5
Nr = 5000
max_theta = np.pi
min_theta = 1e-5
Ntheta = 100

# Grids of points
r_numpy = np.geomspace(min_r, max_r, Nr)
r_grid_1d = torch.tensor(r_numpy, dtype=torch.float64, device=device)
r_grid = r_grid_1d.reshape(-1, 1)
r_grid.requires_grad = True
theta_numpy = np.linspace(min_theta, max_theta, Ntheta)
theta_grid_1d = torch.tensor(theta_numpy, dtype=torch.float64, device=device)
theta_grid = theta_grid_1d.reshape(1, -1)

# List of values for the search (only J)
J_values = [5.70, 5.71, 5.72, 5.73, 5.74, 5.75, 5.76, 5.77, 5.78, 5.79, 5.80]
lam = 0.8

print(f"{'J':<10} | {'Error H*r/J (%)':<20}")
print("-" * 35)

# Variables to register the best parameter
best_error = float('inf')
best_J = None

# Search over the different values of J
for J_test in J_values:
    try:
        # Before the PINN, we find the optimal L for each J
        model_am = AtiyahMantonSkyrmion(initial_L=0.1).to(device)
        optimizer_am = optim.Adam(model_am.parameters(), lr=0.1)
    
        for epoch in range(201):
            optimizer_am.zero_grad()
            current_E, _, current_L, _, _, _ = energy_functional(
                model_am, physics, r_grid, theta_grid, r_grid_1d, theta_grid_1d, lam, J_test
            )
            current_E.backward()
            optimizer_am.step()
    
        _, _, L_tensor, _, _, _ = energy_functional(
            model_am, physics, r_grid, theta_grid, r_grid_1d, theta_grid_1d, lam, J_test
        )
        optimal_L_current = L_tensor.item()
        
        # Training of the PINN for the current J
        ode = ODE().to(device)
        odeloss = ODELoss(ode)
        optimizer_nn = optim.Adam(ode.parameters(), lr=8e-3)
        scheduler = optim.lr_scheduler.ExponentialLR(optimizer_nn, gamma=(0.01)**(1/2000))
        
        for epoch in range(501):
            optimizer_nn.zero_grad()
            current_E_nn, current_r_nn, _, _, _ = odeloss(
                physics, r_grid, theta_grid, r_grid_1d, theta_grid_1d, lam, J_test, optimal_L_current, max_r
            )
            current_E_nn.backward()
            optimizer_nn.step()
            scheduler.step()
            
        # Final values after training the PINN
        E_nucleon_tensor, r_nucleon_tensor, _, _, _ = odeloss(
            physics, r_grid, theta_grid, r_grid_1d, theta_grid_1d, lam, J_test, optimal_L_current, max_r
        )
        E_nucleon = (2 * np.pi**2) * E_nucleon_tensor.item()
        r_nucleon = r_nucleon_tensor.item()

        # Experimental ratio and error (Only H*r/J)
        dimensionsless_product = E_nucleon * r_nucleon / J_test
        product_error = 100 * abs(dimensionsless_product - 3.957) / 3.957
        
        if product_error < best_error:
            best_error = product_error
            best_J = J_test
            best_E = E_nucleon
            best_r = r_nucleon
        
        # Results for the current J
        print(f"{J_test:<10.3f} | {product_error:<20.5f}")
    
    except KeyboardInterrupt:
        break

# Final results
print("-" * 35)
print(f"Best J^* fit: {best_J}")
print(f"Relative error: {best_error:.4f}%")

RB = 0.72 / best_r
f = (197.3/RB)*(np.sqrt(3/4)/best_J)**(1/4)
lam_final = lam*(RB*f/197.3)**2

print("-" * 35)
print(f"RB = {RB:.5f} fm")
print(f"f = {f:.5f} MeV")
print(f"Lambda = {lam_final:.5f}")
print(f"J* = {best_J:.5f} = {best_J/np.sqrt(3/4):.5f} J")
print(f"E^* = {best_E:.5f} | E = {(best_E * f**4 * (RB / 197.3)**3):.5f} MeV")
print(f"r^* = {best_r:.5f} | r = {(best_r * RB):.5f} fm")

J          | Error H*r/J (%)     
-----------------------------------
5.700      | 0.73762             
5.710      | 1.84045             
5.720      | 0.17145             
5.730      | 2.99899             
5.740      | 6.11635             
5.750      | 2.39270             
5.760      | 1.26240             
-----------------------------------
Best J^* fit: 5.72
Relative error: 0.1714%
-----------------------------------
RB = 1.83496 fm
f = 67.07093 MeV
Lambda = 0.31128
J* = 5.72000 = 6.60489 J
E^* = 57.58519 | E = 937.44556 MeV
r^* = 0.39238 | r = 0.72000 fm
